---
title: Parliamentary Handbook API
authors: 
    - Tim Sherratt
draft: true
---

The *Parliamentary Handbook* includes a simple API that you can use to get the data about parliament and parliamentarians in machine-readable form.

Aggregated data about individuals can be accessed via: <https://handbookapi.aph.gov.au/api/individuals/>. However, this data only includes a summary of the available information for each person. There are a variety of other endpoints that can be used to access the full data. For example:

| description | endpoint | parameters |
|-------------|----------|-----------|
| records of service, includes parliamentary terms, committee membership etc | `recordsofservice` | `?filter=PHID eq '[PH ID]'` |
| party membership | `PartiesData/GetPartyServiceForParliamentarian` | `?personId=[PHID]` | |

## Records of service

Records of Service types (in the `ROSTypeID` field):

- `9` parliamentary terms
- `5` ministry appointments
- `22` shadow ministry appointments
- `7` Parliamentary Party Positions
- `2` Committee Service
- `10` Party Positions
- `18` occupations
- `21` qualifications
- `6` Parliamentary Appointments

https://handbookapi.aph.gov.au/api/ministries?Select=MID,DateStart,DateEnd


https://handbookapi.aph.gov.au/api/shadowministries?Select=SMID,DateStart,DateEnd

https://handbookapi.aph.gov.au/api/PartiesData/GetPartyServiceForParliamentarian?personId=N26

## Elections

All federal elections: <https://handbookapi.aph.gov.au/api/Elections/Elections>

### Individual elections

Summary: <https://handbookapi.aph.gov.au/api/Elections/Election?electionId=202>
House results by division: <https://handbookapi.aph.gov.au/api/Elections/Divisions?state=&year=1901>
Members elected: <https://handbookapi.aph.gov.au/api/Elections/MembersElected?year=1901>
Senators elected: <https://handbookapi.aph.gov.au/api/Elections/SenatorsElected?year=1901>

There's also detailed results by division, but the amount of data seems to vary.

## Electorates

All electorates: <https://handbookapi.aph.gov.au/api/electorates?$select=EID,Electorate,State,Established,Ceased,Origin,Notes&apply=$filter=Electorate%20ne%20%27%27>

Each individual electorate has set of maps showing redistributions. The dates are here: <https://handbookapi.aph.gov.au/api/ElectorateMaps/ElectorateMaps?electorate=Adelaide&state=South%20Australia>

But if you request an individual map the geodata is KML encoded as base64.

So it should be possible to compile a complete set of historical electorate maps...

In [24]:
import requests
from pathlib import Path
import time
from decodify import decode_message

response = requests.get("https://handbookapi.aph.gov.au/api/electorates?$select=EID,Electorate,State,Established,Ceased,Origin,Notes&apply=$filter=Electorate%20ne%20%27%27")

In [ ]:
data = response.json()
electorates = data["value"]

for electorate in electorates:
    name = electorate["Electorate"]
    state = electorate["State"]
    response = requests.get("https://handbookapi.aph.gov.au/api/ElectorateMaps/ElectorateMaps", params={"electorate": name, "state": state})
    maps = response.json()
    for emap in maps:
        response = requests.get("https://handbookapi.aph.gov.au/api/ElectorateMaps/ElectorateMap", params = {"id": emap["Id"]})
        geo_data = response.json()
        kml, probabilities = decode_message(geo_data["GeoData"])
        file_name = f"{name}_{state.replace(" ", "-")}_{emap["Id"]}_{emap["DateFrom"]}-{emap["DateTo"]}.kml"
        Path("kml", file_name).write_text(kml)
        time.sleep(5)
    time.sleep(10)